# Final Project - High-Accuracy Overnight YOLO Road Detection

This is the accuracy-focused copy of `01_YOLO_BDD100K_Training.ipynb`. The trained
original is intentionally unchanged. This notebook reuses the strongest available
checkpoint, adds class-imbalance handling and a high-resolution refinement pass, then
performs held-out testing, measured confidence calibration, and deployment benchmarking.


## What changes from the baseline

The recorded baseline used YOLO11n, 2% of the training set, and 13 short CPU epochs.
It reached validation F1 0.389 and mAP@50 0.329 while still improving at the final
epoch. The default profile here is measured for the available 4 GB RTX 3050:

- Continue the already-adapted YOLO11n checkpoint instead of discarding its training.
- A deterministic, class-aware 4,000-image main subset instead of 1,400 images.
- Partial inverse-frequency classification weighting for rare buses and trucks.
- A 768-pixel refinement pass on a more strongly balanced 1,500-image subset.
- Automatic comparison of main and refined checkpoints so refinement is kept only
  when validation results improve.
- Validation-derived per-class thresholds targeting 75% precision for live detection.

Object detection has no single classification accuracy. The acceptance table reports
precision, recall, F1, mAP@50, and mAP@50:95. A 0.70-0.80 F1 or mAP@50 result is an
ambitious measured target, not something a threshold can guarantee.


In [ ]:
from pathlib import Path
from collections import Counter
import json
import math
import random
import shutil
import subprocess
import sys
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import Image as DisplayImage, display
from tqdm.auto import tqdm
from ultralytics import YOLO

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_float32_matmul_precision("high")

print(f"Project: {PROJECT_ROOT}")
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__} | CUDA build: {torch.version.cuda}")


## Select a run profile

`overnight_gpu` is the recommended local profile. Its two training stages have an
11-hour combined ceiling and save checkpoints every five epochs. `gpu_smoke` is for
pipeline testing only. `nextgen_gpu` starts fresh from YOLO26s and needs a much longer
run on this low-power laptop GPU. `full_accuracy` is intended for a larger GPU. The
CPU fallback is functional but is not expected to reach the requested target.

Set `RUN_TAG` to a new value for a fresh experiment. To continue an interrupted
stage, leave the tag unchanged and set the matching `RESUME_*` switch to `True`.
For another full overnight pass, set `START_WEIGHTS_OVERRIDE` to the prior selected
checkpoint and use a new tag.


In [ ]:
RUN_MODE = "overnight_gpu"  # gpu_smoke | overnight_gpu | nextgen_gpu | full_accuracy | cpu_fallback
RUN_TAG = "v2"

RUN_MAIN_TRAINING = True
RUN_REFINEMENT = True
RESUME_MAIN = False
RESUME_REFINEMENT = False
USE_TRAINED_BASELINE = True
START_WEIGHTS_OVERRIDE = None  # Example: r"runs/.../weights/best.pt"

TARGET_F1 = 0.75
TARGET_DEPLOY_PRECISION = 0.75
MIN_DEPLOY_RECALL = 0.15
MIN_DISPLAY_CONFIDENCE = 0.60
EXPORT_ONNX = False
EXPORT_TENSORRT = False

PROFILES = {
    "gpu_smoke": dict(
        model="yolo26n.pt", family="YOLO26", device=0, workers=2,
        main_images=1000, main_imgsz=640, main_batch=4,
        main_epochs=1, main_hours=None,
        refine_images=500, refine_imgsz=640, refine_batch=4,
        refine_epochs=1, refine_hours=None, eval_batch=4,
    ),
    "overnight_gpu": dict(
        model="yolo26n.pt", family="YOLO26", device=0, workers=4,
        main_images=4000, main_imgsz=704, main_batch=0.70,
        main_epochs=100, main_hours=8.0,
        refine_images=1500, refine_imgsz=768, refine_batch=0.70,
        refine_epochs=50, refine_hours=3.0, eval_batch=4,
    ),
    "nextgen_gpu": dict(
        model="yolo26s.pt", family="YOLO26", device=0, workers=4,
        main_images=6000, main_imgsz=768, main_batch=0.70,
        main_epochs=100, main_hours=24.0,
        refine_images=2500, refine_imgsz=832, refine_batch=0.70,
        refine_epochs=50, refine_hours=10.0, eval_batch=2,
    ),
    "full_accuracy": dict(
        model="yolo26m.pt", family="YOLO26", device=0, workers=6,
        main_images=None, main_imgsz=832, main_batch=0.65,
        main_epochs=100, main_hours=None,
        refine_images=30000, refine_imgsz=960, refine_batch=0.65,
        refine_epochs=30, refine_hours=None, eval_batch=2,
    ),
    "cpu_fallback": dict(
        model="yolo26n.pt", family="YOLO26", device="cpu", workers=0,
        main_images=6000, main_imgsz=640, main_batch=4,
        main_epochs=50, main_hours=8.0,
        refine_images=2500, refine_imgsz=704, refine_batch=2,
        refine_epochs=25, refine_hours=2.0, eval_batch=4,
    ),
}

cfg = PROFILES[RUN_MODE]
cfg = dict(cfg)
BASELINE_WEIGHTS = (
    PROJECT_ROOT / "runs" / "notebooks" / "yolo"
    / "bdd100k_cpu_quick_finetuned" / "weights" / "best.pt"
)
if START_WEIGHTS_OVERRIDE:
    override_path = Path(START_WEIGHTS_OVERRIDE)
    if not override_path.is_absolute():
        override_path = PROJECT_ROOT / override_path
    cfg["model"] = str(override_path)
elif RUN_MODE == "overnight_gpu" and USE_TRAINED_BASELINE and BASELINE_WEIGHTS.exists():
    cfg["model"] = str(BASELINE_WEIGHTS)
    cfg["family"] = "YOLO11"

RUN_ROOT = PROJECT_ROOT / "runs" / "notebooks" / "yolo_accuracy"
MANIFEST_ROOT = RUN_ROOT / "manifests" / RUN_TAG
DATA_YAML = PROJECT_ROOT / "data" / "bdd100k_yolo" / "data.yaml"
MAIN_NAME = f"bdd100k_{RUN_MODE}_{RUN_TAG}_main"
REFINE_NAME = f"bdd100k_{RUN_MODE}_{RUN_TAG}_refine"

print(f"Starting checkpoint: {cfg['model']}")
cfg


## Verify GPU acceleration

The RTX 3050 requires a CUDA-enabled PyTorch wheel. If this cell reports a CPU-only
build, close Jupyter, run the command below in the activated environment, reopen
Jupyter, and restart the kernel:

```powershell
C:\tf214_hw2\Scripts\python.exe -m pip install --force-reinstall `
  torch==2.10.0 torchvision==0.25.0 `
  --index-url https://download.pytorch.org/whl/cu126
```


In [ ]:
if cfg["device"] != "cpu" and not torch.cuda.is_available():
    raise RuntimeError(
        "This profile requires CUDA, but the current notebook kernel has CPU-only "
        "PyTorch. Restart Jupyter after installing the CUDA wheels shown above."
    )

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu.name} | VRAM: {gpu.total_memory / 1024**3:.1f} GB")
    x = torch.randn(1024, 1024, device="cuda")
    _ = x @ x
    torch.cuda.synchronize()
    del x
    torch.cuda.empty_cache()
else:
    print("CPU fallback selected. Expect materially lower accuracy per hour.")


## Verify the full train/validation/test dataset


In [ ]:
assert DATA_YAML.exists(), f"Missing {DATA_YAML}. Convert BDD100K before training."
data_config = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))
dataset_root = Path(data_config["path"])
class_names = (
    list(data_config["names"].values())
    if isinstance(data_config["names"], dict)
    else list(data_config["names"])
)

split_rows = []
for split in ("train", "val", "test"):
    image_dir = dataset_root / "images" / split
    label_dir = dataset_root / "labels" / split
    split_rows.append({
        "split": split,
        "images": sum(1 for _ in image_dir.glob("*.jpg")),
        "label_files": sum(1 for _ in label_dir.glob("*.txt")),
    })

split_table = pd.DataFrame(split_rows)
display(split_table)
assert split_table.set_index("split").loc[["train", "val", "test"], "images"].min() > 0
assert split_table.loc[split_table["split"] == "train", "images"].iloc[0] >= 70000


## Build deterministic class-aware training manifests

The previous `fraction=0.02` setting selected the first 1,400 sorted images. Here,
weighted sampling without replacement keeps broad scene diversity while increasing
the chance that images containing rare classes enter the time-bounded run. The main
sample uses mild weighting; the refinement sample uses stronger weighting.


In [ ]:
train_image_dir = dataset_root / "images" / "train"
train_label_dir = dataset_root / "labels" / "train"
label_paths = sorted(train_label_dir.glob("*.txt"))

records = []
box_counts = Counter()
image_counts = Counter()
for label_path in tqdm(label_paths, desc="Reading training labels"):
    classes = []
    for line in label_path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        class_id = int(line.split()[0])
        classes.append(class_id)
        box_counts[class_id] += 1
    present = frozenset(classes)
    for class_id in present:
        image_counts[class_id] += 1
    image_path = train_image_dir / f"{label_path.stem}.jpg"
    if image_path.exists():
        records.append((image_path, present))

distribution = pd.DataFrame([
    {
        "class_id": class_id,
        "class": class_names[class_id],
        "training_images": image_counts[class_id],
        "training_boxes": box_counts[class_id],
    }
    for class_id in range(len(class_names))
])
display(distribution)
assert len(records) >= 70000, f"Expected 70,000 training records, found {len(records)}"


In [ ]:
def weighted_sample(records, count, exponent, seed):
    if count is None or count >= len(records):
        return list(records)
    largest = max(image_counts.values())
    class_weight = {
        class_id: (largest / max(1, image_counts[class_id])) ** exponent
        for class_id in range(len(class_names))
    }
    rng = random.Random(seed)
    keyed = []
    for record in records:
        _, present = record
        weight = max((class_weight[c] for c in present), default=1.0)
        key = rng.random() ** (1.0 / weight)
        keyed.append((key, record))
    return [record for _, record in sorted(keyed, key=lambda item: item[0], reverse=True)[:count]]


def write_manifest_and_yaml(name, selected):
    MANIFEST_ROOT.mkdir(parents=True, exist_ok=True)
    manifest_path = MANIFEST_ROOT / f"{name}.txt"
    manifest_path.write_text(
        "\n".join(path.resolve().as_posix() for path, _ in selected) + "\n",
        encoding="utf-8",
    )
    variant = dict(data_config)
    variant["path"] = dataset_root.resolve().as_posix()
    variant["train"] = manifest_path.resolve().as_posix()
    variant["val"] = (dataset_root / "images" / "val").resolve().as_posix()
    variant["test"] = (dataset_root / "images" / "test").resolve().as_posix()
    yaml_path = MANIFEST_ROOT / f"{name}.yaml"
    yaml_path.write_text(yaml.safe_dump(variant, sort_keys=False), encoding="utf-8")
    return manifest_path, yaml_path


main_records = weighted_sample(records, cfg["main_images"], exponent=0.25, seed=SEED)
refine_records = weighted_sample(records, cfg["refine_images"], exponent=0.65, seed=SEED + 1)
MAIN_MANIFEST, MAIN_DATA_YAML = write_manifest_and_yaml("main", main_records)
REFINE_MANIFEST, REFINE_DATA_YAML = write_manifest_and_yaml("refine", refine_records)

def manifest_distribution(selected):
    counts = Counter()
    for _, present in selected:
        for class_id in present:
            counts[class_id] += 1
    return pd.DataFrame([
        {"class": class_names[i], "images_with_class": counts[i]}
        for i in range(len(class_names))
    ])

print(f"Main manifest: {len(main_records):,} images")
display(manifest_distribution(main_records))
print(f"Refinement manifest: {len(refine_records):,} images")
display(manifest_distribution(refine_records))


## Baseline recorded before this upgrade

These values come from the unchanged notebook and make the improvement ratio explicit.


In [ ]:
BASELINE = {
    "precision": 0.4438674020,
    "recall": 0.3465878195,
    "f1": 0.3892416188,
    "mAP50": 0.3286961061,
    "mAP50_95": 0.1725033497,
}
display(pd.DataFrame([BASELINE], index=["YOLO11n CPU quick baseline"]).style.format("{:.3f}"))


## Training helpers and restart protection

Fresh runs refuse to reuse an existing directory. This prevents accidentally replacing
an overnight checkpoint. Set the matching `RESUME_*` flag after an interruption, or
change `RUN_TAG` for a new experiment.


In [ ]:
def train_stage(start_weights, data_yaml, run_name, stage, resume=False):
    run_dir = RUN_ROOT / run_name
    last_weights = run_dir / "weights" / "last.pt"
    if resume:
        assert last_weights.exists(), f"No resumable checkpoint at {last_weights}"
        print(f"Resuming {stage} from {last_weights}")
        return YOLO(str(last_weights)).train(resume=True)
    if run_dir.exists():
        raise FileExistsError(
            f"{run_dir} already exists. Set the {stage.upper()} resume switch or change RUN_TAG."
        )

    is_main = stage == "main"
    hours = cfg["main_hours"] if is_main else cfg["refine_hours"]
    train_args = dict(
        data=str(data_yaml),
        epochs=cfg["main_epochs"] if is_main else cfg["refine_epochs"],
        imgsz=cfg["main_imgsz"] if is_main else cfg["refine_imgsz"],
        batch=cfg["main_batch"] if is_main else cfg["refine_batch"],
        device=cfg["device"],
        workers=cfg["workers"],
        project=str(RUN_ROOT),
        name=run_name,
        exist_ok=False,
        pretrained=True,
        optimizer="AdamW",
        patience=15 if is_main else 10,
        cos_lr=True,
        warmup_epochs=1.0,
        cls_pw=0.35 if is_main else 0.55,
        box=7.5,
        cls=0.50,
        mosaic=0.70 if is_main else 0.40,
        mixup=0.02 if is_main else 0.0,
        hsv_h=0.015,
        hsv_s=0.45 if is_main else 0.35,
        hsv_v=0.45,
        degrees=0.0,
        translate=0.10 if is_main else 0.06,
        scale=0.60 if is_main else 0.35,
        fliplr=0.50,
        close_mosaic=8 if is_main else 5,
        amp=cfg["device"] != "cpu",
        channels_last=cfg["device"] != "cpu",
        cache=False,
        max_det=300,
        seed=SEED,
        deterministic=True,
        plots=True,
        save=True,
        save_period=5,
        val=True,
    )
    if is_main:
        train_args.update(lr0=6e-4, lrf=0.05, weight_decay=6e-4)
    else:
        train_args.update(lr0=3e-4, lrf=0.05, weight_decay=7e-4)
    if hours is not None:
        train_args["time"] = hours
    return YOLO(str(start_weights)).train(**train_args)


## Stage 1 - high-capacity main training

On this laptop the default starts from the best checkpoint produced by the unchanged
baseline notebook. That preserves its learned BDD100K features and spends the overnight
budget improving them on more diverse, class-aware data. If that checkpoint is absent,
the profile falls back to pretrained YOLO26n. Mixed precision and channels-last memory
layout reduce GPU memory and training time.


In [ ]:
MAIN_RUN_DIR = RUN_ROOT / MAIN_NAME
if RUN_MAIN_TRAINING:
    main_result = train_stage(
        cfg["model"], MAIN_DATA_YAML, MAIN_NAME, "main", resume=RESUME_MAIN
    )
MAIN_BEST = MAIN_RUN_DIR / "weights" / "best.pt"
assert MAIN_BEST.exists(), f"Main checkpoint not found: {MAIN_BEST}"
MAIN_BEST


## Stage 2 - high-resolution rare-class refinement

The second pass starts from the best main checkpoint, raises image resolution, applies
stronger class weighting, reduces geometric augmentation, and lowers the learning rate.
This specifically targets buses, trucks, pedestrians, traffic lights, and traffic signs
without changing the real-time model family.


In [ ]:
REFINE_RUN_DIR = RUN_ROOT / REFINE_NAME
if RUN_REFINEMENT:
    refine_result = train_stage(
        MAIN_BEST, REFINE_DATA_YAML, REFINE_NAME, "refine", resume=RESUME_REFINEMENT
    )
REFINE_BEST = REFINE_RUN_DIR / "weights" / "best.pt"
assert REFINE_BEST.exists(), f"Refinement checkpoint not found: {REFINE_BEST}"
REFINE_BEST


## Inspect learning curves before model selection


In [ ]:
for run_dir in (MAIN_RUN_DIR, REFINE_RUN_DIR):
    print(run_dir.name)
    for filename in ("results.png", "BoxPR_curve.png", "confusion_matrix_normalized.png"):
        plot_path = run_dir / filename
        if plot_path.exists():
            display(DisplayImage(filename=str(plot_path)))


## Compare main and refined checkpoints on validation data

Rare-class refinement can trade common-car performance for minority-class recall.
Both checkpoints are therefore evaluated under the same settings. The deployment
checkpoint is selected by a balanced F1/mAP@50 score, never by assumption.


In [ ]:
def summarize_metrics(label, metrics):
    precision = float(metrics.box.mp)
    recall = float(metrics.box.mr)
    f1 = 2 * precision * recall / max(1e-12, precision + recall)
    map50 = float(metrics.box.map50)
    map50_95 = float(metrics.box.map)
    return {
        "candidate": label,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "mAP50": map50,
        "mAP50_95": map50_95,
        "selection_score": 0.60 * f1 + 0.40 * map50,
    }


candidate_paths = {"main": MAIN_BEST, "refined": REFINE_BEST}
candidate_metrics = {}
candidate_rows = []
for label, weights in candidate_paths.items():
    model = YOLO(str(weights))
    metrics = model.val(
        data=str(DATA_YAML),
        split="val",
        imgsz=cfg["refine_imgsz"],
        batch=cfg["eval_batch"],
        device=cfg["device"],
        workers=cfg["workers"],
        conf=0.001,
        iou=0.70,
        max_det=300,
        plots=True,
        project=str(RUN_ROOT),
        name=f"{MAIN_NAME}_{label}_selection_val",
        verbose=False,
    )
    candidate_metrics[label] = metrics
    candidate_rows.append(summarize_metrics(label, metrics))

candidate_table = pd.DataFrame(candidate_rows).sort_values("selection_score", ascending=False)
display(candidate_table.style.format({c: "{:.3f}" for c in candidate_table.columns if c != "candidate"}))

SELECTED_LABEL = candidate_table.iloc[0]["candidate"]
BEST_WEIGHTS = candidate_paths[SELECTED_LABEL]
val_metrics = candidate_metrics[SELECTED_LABEL]
best_model = YOLO(str(BEST_WEIGHTS))
print(f"Selected checkpoint: {SELECTED_LABEL} -> {BEST_WEIGHTS}")


## Held-out test evaluation and improvement ratio


In [ ]:
test_metrics = best_model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=cfg["refine_imgsz"],
    batch=cfg["eval_batch"],
    device=cfg["device"],
    workers=cfg["workers"],
    conf=0.001,
    iou=0.70,
    max_det=300,
    plots=True,
    project=str(RUN_ROOT),
    name=f"{MAIN_NAME}_{SELECTED_LABEL}_test",
    verbose=False,
)

summary = pd.DataFrame([
    {**summarize_metrics("validation", val_metrics), "split": "validation"},
    {**summarize_metrics("test", test_metrics), "split": "test"},
]).drop(columns=["candidate", "selection_score"])

for metric in ("precision", "recall", "f1", "mAP50", "mAP50_95"):
    summary[f"{metric}_vs_baseline"] = summary[metric] / BASELINE[metric]

display(summary.style.format({
    column: "{:.3f}"
    for column in summary.columns
    if column != "split"
}))


In [ ]:
def per_class_table(metrics):
    rows = []
    metric_position = {
        int(class_id): position
        for position, class_id in enumerate(metrics.ap_class_index)
    }
    for class_id, name in best_model.names.items():
        position = metric_position.get(class_id)
        precision = float(metrics.box.p[position]) if position is not None else 0.0
        recall = float(metrics.box.r[position]) if position is not None else 0.0
        rows.append({
            "class": name,
            "precision": precision,
            "recall": recall,
            "f1": 2 * precision * recall / max(1e-12, precision + recall),
            "mAP50": float(metrics.box.ap50[position]) if position is not None else 0.0,
            "mAP50_95": float(metrics.box.maps[class_id]),
        })
    return pd.DataFrame(rows)


print("Validation by class")
display(per_class_table(val_metrics).style.format("{:.3f}", subset=[
    "precision", "recall", "f1", "mAP50", "mAP50_95"
]))
print("Test by class")
display(per_class_table(test_metrics).style.format("{:.3f}", subset=[
    "precision", "recall", "f1", "mAP50", "mAP50_95"
]))


## Calibrate confidence thresholds from validation curves

A displayed confidence score is not the same as test accuracy. This cell chooses a
threshold independently for each class. It prefers operating points with at least
75% validation precision and 15% recall, then applies a 0.60 display-score floor.
If a class cannot satisfy that constraint, its best-F1 point is used and clearly
reported as a fallback.


In [ ]:
def calibrate_thresholds(metrics):
    px = np.asarray(metrics.box.px)
    f1_curve = np.asarray(metrics.box.f1_curve)
    p_curve = np.asarray(metrics.box.p_curve)
    r_curve = np.asarray(metrics.box.r_curve)
    rows = []
    thresholds = {}

    for position, class_id_raw in enumerate(metrics.ap_class_index):
        class_id = int(class_id_raw)
        best_f1_index = int(np.nanargmax(f1_curve[position]))
        valid = np.flatnonzero(
            (p_curve[position] >= TARGET_DEPLOY_PRECISION)
            & (r_curve[position] >= MIN_DEPLOY_RECALL)
        )
        if len(valid):
            selected_index = int(valid[np.nanargmax(f1_curve[position, valid])])
            mode = "precision target"
        else:
            selected_index = best_f1_index
            mode = "best F1 fallback"

        threshold = max(MIN_DISPLAY_CONFIDENCE, float(px[selected_index]))
        thresholds[str(class_id)] = threshold
        rows.append({
            "class_id": class_id,
            "class": best_model.names[class_id],
            "threshold": threshold,
            "estimated_precision": float(np.interp(threshold, px, p_curve[position])),
            "estimated_recall": float(np.interp(threshold, px, r_curve[position])),
            "best_f1": float(f1_curve[position, best_f1_index]),
            "selection": mode,
        })
    return thresholds, pd.DataFrame(rows)


class_thresholds, calibration_table = calibrate_thresholds(val_metrics)
display(calibration_table.style.format({
    "threshold": "{:.3f}",
    "estimated_precision": "{:.3f}",
    "estimated_recall": "{:.3f}",
    "best_f1": "{:.3f}",
}))
INFERENCE_CONFIDENCE = min(class_thresholds.values())
print(f"Low inference threshold before per-class filtering: {INFERENCE_CONFIDENCE:.3f}")


## Save the selected weights and deployment configuration


In [ ]:
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "yolo_accuracy"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
DEPLOY_WEIGHTS = OUTPUT_ROOT / "bdd100k_yolo_best.pt"
DEPLOY_CONFIG = OUTPUT_ROOT / "deployment_config.json"
shutil.copy2(BEST_WEIGHTS, DEPLOY_WEIGHTS)

deployment = {
    "model_family": cfg["family"],
    "selected_stage": SELECTED_LABEL,
    "weights": str(DEPLOY_WEIGHTS.resolve()),
    "imgsz": cfg["refine_imgsz"],
    "inference_confidence": INFERENCE_CONFIDENCE,
    "class_thresholds": class_thresholds,
    "class_names": {str(k): v for k, v in best_model.names.items()},
    "target_validation_precision": TARGET_DEPLOY_PRECISION,
    "minimum_display_confidence": MIN_DISPLAY_CONFIDENCE,
    "validation_metrics": summarize_metrics("validation", val_metrics),
    "test_metrics": summarize_metrics("test", test_metrics),
}
DEPLOY_CONFIG.write_text(json.dumps(deployment, indent=2), encoding="utf-8")
deployment_model = YOLO(str(DEPLOY_WEIGHTS))

print(f"Weights: {DEPLOY_WEIGHTS}")
print(f"Thresholds: {DEPLOY_CONFIG}")
print(
    "Live command:\n"
    f'python -m road_detection.realtime_detect --backend yolo '
    f'--weights "{DEPLOY_WEIGHTS}" --config "{DEPLOY_CONFIG}" --source 0'
)


## Qualitative test predictions using calibrated thresholds


In [ ]:
def calibrated_frame(result):
    frame = result.orig_img.copy()
    for box in result.boxes:
        class_id = int(box.cls[0])
        score = float(box.conf[0])
        if score < class_thresholds.get(str(class_id), MIN_DISPLAY_CONFIDENCE):
            continue
        x1, y1, x2, y2 = [int(value) for value in box.xyxy[0].cpu().tolist()]
        color = ((37 * class_id + 60) % 255, (91 * class_id + 110) % 255, (151 * class_id + 80) % 255)
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        text = f"{deployment_model.names[class_id]} {score:.2f}"
        cv2.putText(frame, text, (x1, max(18, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)


test_images = sorted((dataset_root / "images" / "test").glob("*.jpg"))
sample_images = random.Random(SEED).sample(test_images, min(6, len(test_images)))
prediction_results = deployment_model.predict(
    sample_images,
    imgsz=cfg["refine_imgsz"],
    conf=INFERENCE_CONFIDENCE,
    device=cfg["device"],
    quantize=16 if cfg["device"] != "cpu" else None,
    max_det=300,
    verbose=False,
)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for axis, result, image_path in zip(axes.flat, prediction_results, sample_images):
    axis.imshow(calibrated_frame(result))
    axis.set_title(image_path.name)
    axis.axis("off")
for axis in axes.flat[len(prediction_results):]:
    axis.axis("off")
plt.tight_layout()
plt.show()


## Low-light failure-case review


In [ ]:
def brightness(path):
    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    return float(image.mean()) if image is not None else 255.0


dark_images = sorted(test_images, key=brightness)[:min(6, len(test_images))]
dark_results = deployment_model.predict(
    dark_images,
    imgsz=cfg["refine_imgsz"],
    conf=INFERENCE_CONFIDENCE,
    device=cfg["device"],
    quantize=16 if cfg["device"] != "cpu" else None,
    max_det=300,
    verbose=False,
)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for axis, result, image_path in zip(axes.flat, dark_results, dark_images):
    axis.imshow(calibrated_frame(result))
    axis.set_title(f"{image_path.name} | brightness={brightness(image_path):.0f}")
    axis.axis("off")
for axis in axes.flat[len(dark_results):]:
    axis.axis("off")
plt.suptitle("Calibrated low-light predictions")
plt.tight_layout()
plt.show()


## Real-time inference benchmark


In [ ]:
benchmark_images = test_images[:min(50, len(test_images))]
benchmark_frames = [cv2.imread(str(path)) for path in benchmark_images]
benchmark_frames = [frame for frame in benchmark_frames if frame is not None]
for frame in benchmark_frames[:5]:
    _ = deployment_model.predict(
        frame, imgsz=cfg["refine_imgsz"], conf=INFERENCE_CONFIDENCE,
        device=cfg["device"], quantize=16 if cfg["device"] != "cpu" else None, verbose=False
    )
if torch.cuda.is_available():
    torch.cuda.synchronize()

inference_times = []
started = time.perf_counter()
for frame in benchmark_frames:
    result = deployment_model.predict(
        frame,
        imgsz=cfg["refine_imgsz"],
        conf=INFERENCE_CONFIDENCE,
        device=cfg["device"],
        quantize=16 if cfg["device"] != "cpu" else None,
        max_det=300,
        verbose=False,
    )[0]
    inference_times.append(float(result.speed["inference"]))
if torch.cuda.is_available():
    torch.cuda.synchronize()
elapsed = time.perf_counter() - started
fps = len(benchmark_frames) / max(elapsed, 1e-9)
model_fps = 1000.0 / max(1e-9, float(np.mean(inference_times)))
print(f"Camera-like end-to-end speed: {fps:.2f} FPS on {cfg['device']}")
print(f"Neural-network inference only: {model_fps:.2f} FPS")

report = {
    **deployment,
    "inference_fps": fps,
    "model_inference_fps": model_fps,
    "profile": RUN_MODE,
    "run_tag": RUN_TAG,
    "candidate_comparison": candidate_table.to_dict(orient="records"),
}
(OUTPUT_ROOT / "final_evaluation.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
summary.to_csv(OUTPUT_ROOT / "final_metrics.csv", index=False)
calibration_table.to_csv(OUTPUT_ROOT / "confidence_calibration.csv", index=False)


## Optional optimized exports

The `.pt` model already uses CUDA for the live demo. ONNX is portable and TensorRT
can improve NVIDIA inference speed, but exports may install extra dependencies and
take several minutes. Enable only the format you plan to use.


In [ ]:
exported = {}
if EXPORT_ONNX:
    exported["onnx"] = deployment_model.export(
        format="onnx", imgsz=cfg["refine_imgsz"], dynamic=False,
        simplify=True, half=False
    )
if EXPORT_TENSORRT:
    if not torch.cuda.is_available():
        raise RuntimeError("TensorRT export requires the CUDA profile.")
    exported["engine"] = deployment_model.export(
        format="engine", imgsz=cfg["refine_imgsz"], batch=1,
        dynamic=False, simplify=True, half=True, device=0
    )
exported


## Acceptance decision

Treat validation as the tuning set and the held-out test split as the final unbiased
measurement. Do not repeatedly tune against test results. If the target is still not
met, the next evidence-based step is a longer `full_accuracy` run on a larger GPU,
not an artificially higher display threshold.


In [ ]:
acceptance = summary[["split", "f1", "mAP50"]].copy()
acceptance["target_f1"] = TARGET_F1
acceptance["passed_f1"] = acceptance["f1"] >= TARGET_F1
display(acceptance.style.format({"f1": "{:.3f}", "mAP50": "{:.3f}", "target_f1": "{:.2f}"}))

if acceptance["passed_f1"].all():
    print("The measured validation/test F1 target is met.")
else:
    print(
        "The target is not yet met. Keep these measurements honest; use a longer "
        "full_accuracy run or additional labeled data rather than inflating confidence."
    )
